# 03 - LLM Event Analysis

Uses Claude API to:
1. Classify geopolitical events from the curated database
2. Analyze financial sentiment per market
3. Assess market impact with cross-market transmission
4. Attribute DC events to geopolitical causes

**Note:** Requires ANTHROPIC_API_KEY in `.env` file.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from src.data_collection.conflict_tracker import ConflictEventTracker
from src.llm_pipeline.client import ClaudeClient
from src.llm_pipeline.event_classifier import EventClassifier
from src.llm_pipeline.sentiment_analyzer import FinancialSentimentAnalyzer
from src.llm_pipeline.impact_assessor import MarketImpactAssessor

## 1. Load Events & Test API

In [ ]:
tracker = ConflictEventTracker()
events = tracker.get_combined_geopolitical_events()
print(f'Total events: {len(events)}')

# Test API connection
client = ClaudeClient()
response = client.query('What is 2+2? Reply with just the number.')
print(f'API test: {response}')

## 2. Classify Key Events

In [ ]:
classifier = EventClassifier()

# Classify high-severity events
high_severity = events[events['severity'] >= 8].head(5)

classifications = []
for _, event in high_severity.iterrows():
    result = classifier.classify_article(
        title=event['event'],
        content=event['event'],
        date=str(event['date']),
    )
    result['original_event'] = event['event']
    classifications.append(result)
    print(f"Classified: {event['event'][:50]}...")

pd.DataFrame(classifications)

## 3. Market-Specific Sentiment Analysis

In [ ]:
sentiment_analyzer = FinancialSentimentAnalyzer()

# Analyze sentiment for a key event
test_event = events[events['severity'] == events['severity'].max()].iloc[0]
print(f"Analyzing: {test_event['event']}")

sentiment = sentiment_analyzer.analyze_article(
    title=test_event['event'],
    content=test_event['event'],
    date=str(test_event['date']),
)

print(f"\nOverall sentiment: {sentiment.get('overall_sentiment', 'N/A')}")
print(f"Market-specific:")
for market, data in sentiment.get('market_specific_sentiment', {}).items():
    print(f"  {market}: {data}")

## 4. Impact Assessment

In [ ]:
assessor = MarketImpactAssessor()

# Assess impact of a major tariff event
tariff_event = events[events['category'].str.contains('tariff', na=False)].iloc[0]
print(f"Assessing: {tariff_event['event']}")

impact = assessor.assess_event_impact(tariff_event.to_dict())

print(f"\nImpact Summary: {impact.get('impact_summary', 'N/A')[:200]}")
print(f"Magnitude Estimate: {impact.get('magnitude_estimate', {})}")
print(f"India Analysis: {impact.get('india_specific_analysis', {})}")